# 📊 EDA — Phân Tích Xu Hướng Kinh Doanh (2012–2022)

> **Mục tiêu:** Trực quan hóa toàn diện doanh thu, cơ cấu khách hàng, chất lượng traffic và tỷ lệ chuyển đổi nhằm làm rõ hai giai đoạn tăng trưởng (2012–2018) và suy thoái (2019–2022).

---

## 📁 Cấu Trúc Dữ Liệu
```
project/
├── Data/
│   ├── Analytical/
│   │   └── sales.csv           # Cột: Date, Revenue
│   ├── Transaction/
│   │   └── orders.csv          # Cột: order_date, customer_id
│   └── Operational/
│       └── web_traffic.csv     # Cột: date, sessions
└── EDA_Analysis.ipynb
```

---

## 🗂️ Mục Lục
1. [Import Thư Viện](#1)
2. [Cấu Hình & Helper Functions](#2)
3. [Load Dữ Liệu](#3)
4. [Xử Lý & Tổng Hợp Dữ Liệu](#4)
5. [Trực Quan Hóa — 5 Biểu Đồ](#5)
6. [Xuất Kết Quả](#6)
7. [Tóm Tắt Insight](#7)

## ⚙️ 1. Import Thư Viện <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.ticker import PercentFormatter

# Cấu hình style toàn cục
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style("whitegrid", {
    'axes.grid': True,
    'grid.linestyle': ':',
    'grid.alpha': 0.5
})

print("✅ Import thành công!")

## 🔧 2. Cấu Hình & Helper Functions <a id='2'></a>

In [ ]:
# Formatter hiển thị số tiền theo tiếng Việt
def vn_formatter(x, pos):
    if x >= 1e6:
        return f'{x * 1e-6:,.0f} triệu'
    if x >= 1e3:
        return f'{x * 1e-3:,.0f} nghìn'
    return f'{x:,.0f}'

# Màu sắc nhất quán theo report
COLOR_BLUE    = '#3498DB'   # Golden Era / Doanh thu
COLOR_ORANGE  = '#E67E22'   # Số lượng đơn
COLOR_RED     = '#E74C3C'   # Decline / Cảnh báo
COLOR_DARK    = '#1F618D'   # Khách quay lại (Repeat)
COLOR_LIGHT   = '#A9CCE3'   # Khách mới (New)
COLOR_CR_LINE = '#CB4335'   # Đường Retention Rate

# Mốc cấu trúc gãy
MILESTONE = pd.Timestamp('2019-01-01')

print("✅ Config OK")

## 📥 3. Load Dữ Liệu <a id='3'></a>

In [ ]:
sales   = pd.read_csv('Data/Analytical/sales.csv',       parse_dates=['Date'])
orders  = pd.read_csv('Data/Transaction/orders.csv',      parse_dates=['order_date'])
traffic = pd.read_csv('Data/Operational/web_traffic.csv', parse_dates=['date'])

print(f"Sales   : {sales.shape}")
print(f"Orders  : {orders.shape}")
print(f"Traffic : {traffic.shape}")

In [ ]:
# Preview từng bảng
print("=== SALES ===")
display(sales.head(3))

print("=== ORDERS ===")
display(orders.head(3))

print("=== TRAFFIC ===")
display(traffic.head(3))

## 🔄 4. Xử Lý & Tổng Hợp Dữ Liệu <a id='4'></a>

### 4.1 Doanh thu & Đơn hàng theo tháng

In [ ]:
monthly_rev = sales.resample('MS', on='Date')['Revenue'].sum().reset_index()
monthly_ord = orders.resample('MS', on='order_date').size().reset_index(name='orders')

df_monthly = pd.merge(monthly_rev, monthly_ord, left_on='Date', right_on='order_date')

print(f"df_monthly: {df_monthly.shape}")
display(df_monthly.head(3))

### 4.2 Phân tích Retention — Khách mới vs. Khách quay lại

> **Chú thích định nghĩa:**
> - **Khách mới (New):** Khách thực hiện đơn hàng lần đầu tiên
> - **Khách quay lại (Repeat):** Khách phát sinh đơn hàng sau đơn hàng đầu tiên
> - Một khách có thể thuộc cả hai nhóm trong cùng một năm nếu vừa mua lần đầu vừa mua lặp lại ngay sau đó

In [ ]:
# Xác định lần đặt hàng đầu tiên của mỗi khách
first_orders = (
    orders
    .groupby('customer_id')['order_date']
    .min()
    .reset_index()
    .rename(columns={'order_date': 'first_date'})
)

df_ret = pd.merge(orders, first_orders, on='customer_id')
df_ret['type'] = np.where(
    df_ret['order_date'] == df_ret['first_date'],
    'New', 'Repeat'
)

# Thống kê retention theo năm
retention_stats = (
    df_ret
    .groupby([df_ret['order_date'].dt.year, 'type'])['customer_id']
    .nunique()
    .unstack(fill_value=0)
)
retention_stats['Rate'] = (
    retention_stats['Repeat']
    / (retention_stats['New'] + retention_stats['Repeat'])
    * 100
)

print("Retention Stats (2012–2022):")
display(retention_stats)

### 4.3 Traffic & Đơn hàng theo ngày — Tính CR & Moving Average

In [ ]:
daily_traffic = traffic.groupby('date')['sessions'].sum().reset_index()
daily_orders  = orders.groupby('order_date').size().reset_index(name='orders')

df_daily = pd.merge(
    daily_traffic, daily_orders,
    left_on='date', right_on='order_date',
    how='left'
).fillna(0)

# Conversion Rate hàng ngày
df_daily['CR'] = df_daily['orders'] / (df_daily['sessions'] + 1)

# 30-day Moving Average CR (dùng cho biểu đồ — khớp với report)
df_daily = df_daily.sort_values('date')
df_daily['CR_MA30'] = df_daily['CR'].rolling(window=30, min_periods=1).mean()

# Phân kỳ thời gian
df_daily['Period'] = np.where(
    df_daily['date'].dt.year <= 2018,
    '2013–2018 (Golden Era)',
    '2019–2022 (Decline)'
)

print(f"df_daily: {df_daily.shape}")
display(df_daily.head(3))

## 📈 5. Trực Quan Hóa — 5 Biểu Đồ <a id='5'></a>

In [ ]:
fig = plt.figure(figsize=(18, 30))
gs  = fig.add_gridspec(5, 1, hspace=0.45, height_ratios=[1.1, 0.75, 1, 0.85, 0.85])

### 📌 Biểu đồ 1 — Biến Động Doanh Thu & Quy Mô Đơn Hàng (2012–2022)

> **Insight:** Doanh thu và đơn hàng đạt đỉnh giai đoạn **2016–2018**, sau đó suy thoái mạnh từ 2019.  
> Số đơn hàng sau 2019 giảm hơn **50%**, chỉ dao động 4,000–5,000 đơn/tháng.

In [ ]:
ax1      = fig.add_subplot(gs[0])
ax1_twin = ax1.twinx()

ax1.plot(
    df_monthly['Date'], df_monthly['Revenue'],
    color=COLOR_BLUE, lw=2, marker='o', ms=3, label='Doanh thu hàng tháng'
)
ax1_twin.plot(
    df_monthly['Date'], df_monthly['orders'],
    color=COLOR_ORANGE, lw=1.5, ls='--', marker='s', ms=3, alpha=0.8, label='Số lượng đơn hàng'
)

ax1.set_title(
    'BIẾN ĐỘNG DOANH THU VÀ QUY MÔ ĐƠN HÀNG (2012 - 2022)',
    loc='left', fontsize=14, fontweight='bold', pad=10
)
ax1.set_ylabel('Doanh thu (Triệu)')
ax1_twin.set_ylabel('Số lượng đơn hàng', color=COLOR_ORANGE)
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(vn_formatter))
ax1.axvline(MILESTONE, color=COLOR_RED, ls='-', lw=2, alpha=0.8, label='Điểm gãy 2019')
ax1.fill_betweenx(
    ax1.get_ylim(),
    pd.Timestamp('2019-01-01'), df_monthly['Date'].max(),
    alpha=0.04, color=COLOR_RED
)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

### 📌 Biểu đồ 2 — Seasonal Heatmap: Điểm Nóng Mùa Vụ

> **Insight:** Mô hình mùa vụ lặp lại rõ rệt.  
> - Đỉnh doanh thu tại **Q2** (tháng 4–6) — nhu cầu cao nhất trong năm  
> - Giảm dần vào **Q3**, đáy tại **Q4 và Q1** năm sau

In [ ]:
ax2 = fig.add_subplot(gs[1])

pivot_rev = (
    df_monthly
    .assign(
        Year  = df_monthly['Date'].dt.year,
        Month = df_monthly['Date'].dt.month
    )
    .pivot(index='Year', columns='Month', values='Revenue')
)
# Đổi tên cột tháng
month_labels = ['T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12']
pivot_rev.columns = month_labels

sns.heatmap(
    pivot_rev, cmap='YlGnBu', annot=False,
    ax=ax2, cbar_kws={'label': 'Doanh thu'},
    linewidths=0.3, linecolor='white'
)
ax2.set_title(
    'HEATMAP DOANH THU: NHẬN DIỆN ĐIỂM NÓNG MÙA VỤ QUA CÁC NĂM',
    loc='left', fontsize=14, fontweight='bold', pad=10
)
ax2.set_xlabel('Tháng')
ax2.set_ylabel('Năm')

### 📌 Biểu đồ 3 — Phân Tích Cơ Cấu Khách Hàng & Retention Rate (2012–2022)

> **Insight (từ report):**
> - Tổng khách hàng giảm gần **50%** so với đỉnh 2013 (từ ~48,200 xuống ~24,700 khách)
> - Khách mới giai đoạn 2019–2022 chỉ còn ~**1,200 khách/năm** — giảm gần **20 lần** so với thời kỳ đỉnh cao
> - Tỷ trọng khách cũ **94.7%** là hệ quả thiếu hụt luồng khách mới, không phải thành công giữ chân
> - Doanh nghiệp đang tồn tại dựa trên tệp khách cũ đang thu hẹp — tiến gần đến **"điểm gãy" về quy mô**

In [ ]:
ax3      = fig.add_subplot(gs[2])
ax3_twin = ax3.twinx()

ax3.bar(
    retention_stats.index, retention_stats['Repeat'],
    color=COLOR_DARK, label='Khách hàng quay lại (Repeat)', alpha=0.85
)
ax3.bar(
    retention_stats.index, retention_stats['New'],
    bottom=retention_stats['Repeat'],
    color=COLOR_LIGHT, label='Khách hàng mới (New)', alpha=0.85
)
ax3_twin.plot(
    retention_stats.index, retention_stats['Rate'],
    color=COLOR_CR_LINE, marker='o', lw=2.5, ms=6, label='Tỷ lệ khách cũ (%)'
)

# Annotate tỷ lệ trên đường
for yr, rate in retention_stats['Rate'].items():
    ax3_twin.annotate(
        f'{rate:.1f}%', xy=(yr, rate),
        xytext=(0, 8), textcoords='offset points',
        ha='center', fontsize=7.5, color=COLOR_CR_LINE
    )

ax3.set_title(
    'PHÂN TÍCH CƠ CẤU KHÁCH HÀNG (2012 - 2022)',
    loc='left', fontsize=14, fontweight='bold', pad=10
)
ax3.set_xlabel('Thời gian (Năm)')
ax3.set_ylabel('Số lượng khách hàng (Người)')
ax3_twin.set_ylabel('Tỷ lệ (%)', color=COLOR_CR_LINE)

lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

### 📌 Biểu đồ 4 — Bằng Chứng Suy Giảm Chất Lượng Traffic

> **Insight:**
> - **2013–2018 (Golden Era):** Tương quan thuận bền vững giữa Sessions và Orders — kênh chuyển đổi lành mạnh
> - **2019–2022 (Decline):** Đứt gãy tương quan hoàn toàn. Sessions tăng cao kỷ lục (>50,000) nhưng Orders giảm sâu (<200 đơn/tháng)
> - Vấn đề cốt lõi: **suy giảm nghiêm trọng của Conversion Rate**, không phải thiếu traffic

In [ ]:
ax4 = fig.add_subplot(gs[3])

sns.scatterplot(
    data=df_daily,
    x='sessions', y='orders',
    hue='Period',
    palette={'2013–2018 (Golden Era)': COLOR_BLUE, '2019–2022 (Decline)': COLOR_RED},
    alpha=0.45, s=35, ax=ax4
)
ax4.set_title(
    'BẰNG CHỨNG VỀ SỰ SUY GIẢM CHẤT LƯỢNG TRAFFIC',
    loc='left', fontsize=14, fontweight='bold', pad=10
)
ax4.set_xlabel('Số Phiên Truy Cập (Sessions)')
ax4.set_ylabel('Số Đơn Hàng (Orders)')
ax4.legend(title='Giai đoạn', fontsize=9)

### 📌 Biểu đồ 5 — Conversion Rate Trend (30-day Moving Average)

> **Insight:**
> - CR duy trì ổn định **0.8%–1.4%** giai đoạn 2013–2018
> - Sau điểm gãy cấu trúc 2019: CR sụp đổ xuống dưới **0.4%** và không phục hồi
> - Đây là nguyên nhân cốt lõi khiến đơn hàng giảm dù traffic vẫn cao

In [ ]:
ax5 = fig.add_subplot(gs[4])

# 30-day Moving Average CR (khớp với biểu đồ trong report)
ax5.plot(
    df_daily['date'], df_daily['CR_MA30'] * 100,
    color=COLOR_RED, lw=1.8, label='30-day Moving Avg CR'
)
# Đường đứt nét: điểm gãy cấu trúc 2019
ax5.axvline(
    MILESTONE, color='black', ls='--', lw=1.5, label='Structural Break (2019)'
)
# Shade hai giai đoạn
ax5.axvspan(
    df_daily['date'].min(), MILESTONE,
    alpha=0.05, color=COLOR_BLUE, label='Golden Era'
)
ax5.axvspan(
    MILESTONE, df_daily['date'].max(),
    alpha=0.05, color=COLOR_RED, label='Decline'
)

ax5.set_title(
    'CONVERSION RATE TREND (30-DAY MOVING AVERAGE)',
    loc='left', fontsize=14, fontweight='bold', pad=10
)
ax5.set_xlabel('Thời gian')
ax5.set_ylabel('Conversion Rate')
ax5.yaxis.set_major_formatter(PercentFormatter(decimals=2))
ax5.legend(fontsize=9)

## 🖨️ 6. Xuất Kết Quả <a id='6'></a>

In [ ]:
import os
os.makedirs('output', exist_ok=True)

plt.savefig('output/EDA_business_analysis_2012_2022.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Đã lưu vào output/EDA_business_analysis_2012_2022.png")

## 📌 7. Tóm Tắt Insight <a id='7'></a>

| # | Biểu đồ | Insight chính |
|---|---------|---------------|
| 1 | Doanh thu & Đơn hàng | Đỉnh 2016–2018; số đơn sau 2019 giảm >50%, chỉ còn 4,000–5,000 đơn/tháng |
| 2 | Seasonal Heatmap | Đỉnh Q2 (T4–T6); đáy Q4–Q1 — mô hình lặp lại rõ rệt qua các năm |
| 3 | Cơ cấu khách hàng | Tổng khách giảm ~50% từ đỉnh 2013; khách mới chỉ ~1,200/năm (giảm 20x); tỷ trọng khách cũ 94.7% là dấu hiệu nguy hiểm |
| 4 | Traffic vs Orders | Đứt gãy tương quan từ 2019: sessions >50,000 nhưng orders <200 — chất lượng traffic sụp đổ |
| 5 | Conversion Rate | CR sụt từ 0.8–1.4% xuống dưới 0.4% sau điểm gãy 2019, không phục hồi |

---

### 🎯 Kết Luận Tổng Thể

> Doanh nghiệp đang trong giai đoạn **suy thoái cấu trúc**, không còn duy trì được quy mô như quá khứ.  
> Vấn đề **không nằm ở traffic** (vẫn cao) mà ở **khả năng chuyển đổi** và **năng lực thu hút khách mới**.  
> Tệp khách cũ đang thu hẹp dần — nếu không can thiệp, doanh nghiệp tiến gần đến **"điểm gãy" về quy mô**.